In [ ]:
import os
import json
from glob import glob
from collections import defaultdict

import numpy as np
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import models, layers, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

import matplotlib.pyplot as plt

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4

NON_DINO_FOLDER = "dataset/not_dinosaur"

MODEL_PATH = "models/stage2_non_dino_species.keras"
CLASS_MAP_PATH = "models/stage2_non_dino_classes.json"

In [ ]:
X, y = [], []
class_to_idx = {}
idx_to_class = {}

def get_base_class(folder_name):
    return folder_name.split("_")[0]

grouped_folders = defaultdict(list)

In [ ]:
for sub in os.listdir(NON_DINO_FOLDER):
    full_path = os.path.join(NON_DINO_FOLDER, sub)
    if os.path.isdir(full_path):
        base_class = get_base_class(sub)
        grouped_folders[base_class].append(full_path)

class_names = sorted(grouped_folders.keys())
class_to_idx = {cls: i for i, cls in enumerate(class_names)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

In [ ]:
print("Визначенні класи:")
for cls, paths in grouped_folders.items():
    print(f"  {cls}: {len(paths)} папки")

In [ ]:
for cls, folders in grouped_folders.items():
    label = class_to_idx[cls]
    for folder in folders:
        paths = glob(f"{folder}/**/*.*", recursive=True)
        for path in tqdm(paths, desc=f"Loading {cls}"):
            try:
                img = Image.open(path).convert("RGB").resize(IMG_SIZE)
                X.append(np.array(img))
                y.append(label)
            except (UnidentifiedImageError, OSError):
                pass

X = np.array(X, dtype="float32")
y = np.array(y, dtype=np.int32)

print("\Загальна кількість картинок:", len(X))
print("Класи:", class_to_idx)

In [ ]:
class_counts = {cls: 0 for cls in class_to_idx.keys()}

for label in y:
    class_name = idx_to_class[label]
    class_counts[class_name] += 1

labels = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(10, 5))
plt.bar(labels, counts, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.title("Баланс класів — Dino Species (Stage 2A)")
plt.ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("models", exist_ok=True)
with open(CLASS_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(idx_to_class, f, indent=2)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [ ]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_flow = train_gen.flow(X_train, y_train, batch_size=BATCH_SIZE)
val_flow = val_gen.flow(X_val, y_val, batch_size=BATCH_SIZE)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

In [ ]:
from tensorflow.keras import layers, models, optimizers

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

# Заморожуємо більшість шарів
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

In [ ]:
model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)
]

In [ ]:
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
print("\nТест:")
model.evaluate(X_test, y_test, verbose=2)

In [ ]:
model.save(MODEL_PATH)